# Triển khai trên Google Colab – Version 0 (CPU tuần tự)

V0 chỉ chạy trên CPU nên build bằng `g++`, không cần GPU. Chọn `Runtime → Run all`.

- **Mã nguồn:** tải thư mục repo (chứa `src/`) từ link Google Drive chung của nhóm (`REPO_DRIVE_URL`, chia sẻ *Anyone with the link*) về `/content/repo`, rồi build từ `src/`. Khi được báo cáo gọi, notebook dùng luôn mã nguồn báo cáo đã tải.
- **Dataset:** tải thẳng từ thư mục Google Drive bằng `gdown` (thư mục phải chia sẻ *Anyone with the link*), lưu vào `/content/dataset`.
- **Dùng từ báo cáo:** `Group07_Midterm.ipynb` chạy notebook này bằng `%run` với `EMBEDDED = True`. Khi đó notebook chỉ tính toán và lưu kết quả; bảng và hình được báo cáo hiển thị ở từng mục bằng các hàm `show_*`. Mở notebook này trực tiếp thì mọi kết quả hiển thị ngay tại chỗ.
- `PAIR_DIR`: đường dẫn tới **thư mục** của một cặp ảnh. Chương trình tự lấy 2 file `*.png` trong thư mục đó, sắp theo tên: file đứng trước là ảnh trái (A), file đứng sau là ảnh phải (B).

In [1]:
DATASET_URL = "https://drive.google.com/drive/folders/1eWi9OBGeIYzuW7wOhOMybv0-ZHNS6YrO"
RUNS = 5                                       # số lần đo (+1 lần warm-up)
REPO_DRIVE_URL = globals().get("REPO_DRIVE_URL", "")   # link thư mục Drive chung của nhóm chứa src/
REPO_DIR = globals().get("REPO_DIR", "")      # đã có khi được báo cáo gọi; để trống: tải từ REPO_DRIVE_URL
EMBEDDED = globals().get("EMBEDDED", False)    # True khi Group07_Midterm.ipynb chạy notebook này bằng %run

## 1. Lấy mã nguồn (src/) và dữ liệu

In [2]:
import glob, os, re, subprocess, time
from IPython.display import Image, display
%cd -q /content

def download_repo(url, out="/content/repo"):
    """Tải thư mục repo từ Google Drive về out (bỏ qua nếu đã tải), trả về thư mục chứa src/."""
    import gdown
    assert url, "Chưa điền REPO_DRIVE_URL"
    find = lambda: sorted(glob.glob(os.path.join(out, "**", "src", "main.cu"), recursive=True))
    if not find():
        print("Đang tải mã nguồn từ Google Drive...")
        gdown.download_folder(url, output=out, quiet=True)
    assert find(), f"Thư mục Drive không có src/main.cu: {url}"
    return os.path.dirname(os.path.dirname(find()[0]))

REPO_DIR = REPO_DIR or download_repo(REPO_DRIVE_URL)
print(f"Mã nguồn: {REPO_DIR}/src")

def download_dataset(url, out="dataset", tries=6):
    """Tải thư mục dataset từ Google Drive. Chỉ dùng gdown để lấy danh sách file, rồi tải
    từng file một và thử lại với thời gian chờ tăng dần: khi tải dồn dập nhiều file, Google
    tạm chặn ("many accesses") khiến `gdown --folder` dừng giữa chừng dù link đã chia sẻ công khai."""
    import gdown, requests
    done = os.path.join(out, ".complete")          # chỉ tạo khi đã tải đủ mọi file
    if os.path.isfile(done):
        print("Dataset đã có trong /content/dataset, bỏ qua bước tải.")
        return
    print("Đang tải dataset từ Google Drive...")
    files = gdown.download_folder(url, output=out, quiet=True, skip_download=True)
    s = requests.Session()
    for f in files:
        if os.path.isfile(f.local_path):           # đã tải ở lần chạy trước
            continue
        os.makedirs(os.path.dirname(f.local_path), exist_ok=True)
        for k in range(1, tries + 1):
            try:
                r = s.get("https://drive.usercontent.google.com/download",
                          params={"id": f.id, "export": "download", "confirm": "t"}, timeout=60)
                if r.ok and "text/html" not in r.headers.get("Content-Type", ""):
                    with open(f.local_path + ".part", "wb") as fp:
                        fp.write(r.content)
                    os.replace(f.local_path + ".part", f.local_path)
                    break
                err = f"HTTP {r.status_code}"
            except requests.RequestException as e:
                err = type(e).__name__
            if k == tries:
                raise RuntimeError(f"Không tải được {f.path} ({err}): kiểm tra quyền chia sẻ "
                                   "(Anyone with the link) hoặc đợi vài phút rồi chạy lại cell")
            print(f"  {f.path}: {err}, thử lại sau {5 * 2 ** (k - 1)} giây...")
            time.sleep(5 * 2 ** (k - 1))
    open(done, "w").close()

download_dataset(DATASET_URL)

PAIR_DIRS = sorted(d for d in {os.path.dirname(p) for p in glob.glob("/content/dataset/**/*.png", recursive=True)}
                   if len(glob.glob(os.path.join(d, "*.png"))) == 2)
print(f"  -> Xong: {len(PAIR_DIRS)} cặp ảnh trong /content/dataset")

## 2. Build chương trình

In [ ]:
!g++ -O3 -std=c++17 -x c++ "{REPO_DIR}/src/main.cu" -x none "{REPO_DIR}"/src/*.cpp -o stitch && echo "BUILD OK: /content/stitch"

## 3. Ghép cặp chỉ định

`pair_images` lấy 2 file `*.png` trong `PAIR_DIR` (báo lỗi nếu thư mục không có đúng 2 ảnh), rồi gọi
`./stitch <trái> <phải> <output> --version 0 --runs RUNS`. Cell hiển thị ảnh trái và ảnh phải, ảnh panorama **không crop** (chạy thêm 1 lần với `--no-crop`, giữ vùng đen để thấy hình dạng hai ảnh sau khi warp), ảnh panorama **có crop**, rồi bảng thời gian từng stage (đo trên lần chạy có crop).

In [ ]:
import base64, io
from IPython.display import HTML
from PIL import Image as PILImage

PAIR_DIR = "/content/dataset/medium/standard pairs/Outdoor/Pair 1"   # thư mục chứa 2 ảnh .png cần ghép

def pair_images(pair_dir):
    imgs = sorted(glob.glob(os.path.join(pair_dir, "*.png")))
    assert len(imgs) == 2, f"'{pair_dir}' cần có đúng 2 ảnh .png (đang có {len(imgs)})"
    return imgs

def stitch(pair_dir, out, runs=RUNS, crop=True):
    left, right = pair_images(pair_dir)
    cmd = ["./stitch", left, right, out, "--version", "0", "--runs", str(runs)]
    return subprocess.run(cmd if crop else cmd + ["--no-crop"], capture_output=True, text=True)

def figure(path, max_w=1000, caption=None):
    # Thu nhỏ thành JPEG trong bộ nhớ để notebook không bị nặng với ảnh 4K
    img = PILImage.open(path).convert("RGB")
    size = f"{img.width}×{img.height}"
    img.thumbnail((max_w, max_w))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=90)
    src = base64.b64encode(buf.getvalue()).decode()
    return (f'<figure style="flex:1; margin:0; text-align:center">'
            f'<img src="data:image/jpeg;base64,{src}" style="width:100%; height:auto">'
            f'<figcaption>{caption or os.path.basename(path)} ({size})</figcaption></figure>')

if not EMBEDDED:   # chỉ để minh họa khi mở notebook trực tiếp, báo cáo không dùng
    left, right = pair_images(PAIR_DIR)
    print("Ảnh trái và ảnh phải:")
    display(HTML(f'<div style="display:flex; gap:32px; max-width:1400px">{figure(left)}{figure(right)}</div>'))

    p = stitch(PAIR_DIR, "/content/panorama.jpg")                                # đo thời gian (có crop)
    q = stitch(PAIR_DIR, "/content/panorama_nocrop.jpg", runs=1, crop=False)     # chỉ để xem toàn bộ canvas
    if p.returncode == 0 and q.returncode == 0:
        print("\nẢnh panorama (không crop, giữ vùng đen):")
        display(HTML(f'<div style="max-width:1400px">{figure("/content/panorama_nocrop.jpg", 1600, "Không crop")}</div>'))
        print("\nẢnh panorama (crop về hình chữ nhật lớn nhất không có pixel đen):")
        display(HTML(f'<div style="max-width:1400px">{figure("/content/panorama.jpg", 1600, "Có crop")}</div>'))
        lines = p.stdout.splitlines()
        print("\n".join(lines[next(i for i, l in enumerate(lines) if l.startswith("Image size")):]))
    else:
        print(p.stdout, p.stderr, q.stdout, q.stderr)

## 4. Kiểm tra độ chính xác với ground truth (cặp tổng hợp GT1)

```
synthetic/
├── GT1.jpg                    panorama gốc (ground truth)
├── small/GT1/A.png, B.png     A, B đã resize 1280×720
├── medium/GT1/A.png, B.png    1920×1080
└── large/GT1/A.png, B.png     3840×2160
```

A và B là hai vùng cắt theo chiều ngang của cùng panorama `GT1.jpg` (mỗi vùng phủ hết chiều cao), sau đó được resize giống các cặp khác. Vì vậy homography thật ánh xạ B → A là một **phép tịnh tiến thuần**:

$$H_{gt} = \begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}$$

**Cách xác định $t$:**
1. Dò vị trí của A và B trong `GT1.jpg` bằng SSD vét cạn. Kết quả chỉ là ước lượng thô vì `GT1.jpg` nhỏ hơn A, B khoảng 2–6 lần.
2. Tinh chỉnh tới mức dưới pixel bằng **căn chỉnh dày đặc** trên toàn bộ vùng chồng lấp giữa A và B: tìm SSD theo pixel nguyên, rồi lặp Lucas–Kanade. Bước này dùng mọi pixel nên độc lập với pipeline keypoint của V0.

**Độ đo** (tính trên các điểm của ảnh B, đơn vị là pixel của ảnh A):
- **ACE** (Average Corner Error): trung bình của $\lVert H_{V0}\,c - H_{gt}\,c \rVert$ trên 4 góc $c$ của ảnh B.
- **Max corner:** sai số lớn nhất trong 4 góc trên.
- **RMSE overlap:** căn bậc hai trung bình bình phương sai số trên lưới 160×90 điểm của B, chỉ lấy những điểm mà $H_{gt}$ chiếu vào bên trong A, tức vùng thực sự được ghép.
- **Inlier / Match:** số inlier của RANSAC trên số match sau ratio test và cross-check.

In [ ]:
import numpy as np
import pandas as pd

STAGES = ["1. Preprocessing", "2. Keypoint detection", "3. Description", "4. Matching",
          "5. Homography (RANSAC)", "6. Warping & blending", "Total (end-to-end)"]
RESOLUTIONS = ["small", "medium", "large"]

def parse_run(stdout):
    """Đọc thống kê của lần warm-up, ma trận H (B -> A) và thời gian trung bình từng stage (ms) từ output của ./stitch."""
    run = {}
    if m := re.search(r"(\d+) keypoints in A, (\d+) in B", stdout):
        run["kp_A"], run["kp_B"] = int(m[1]), int(m[2])
    if m := re.search(r"(\d+) inliers / (\d+) matches", stdout):
        run["inliers"], run["matches"] = int(m[1]), int(m[2])
    if m := re.search(r"H \(B -> A\)\s*: \[([^\]]+)\]", stdout):
        run["H"] = np.array(m[1].replace(";", " ").split(), dtype=float).reshape(3, 3)
    for name, avg in re.findall(r"^(\d\. .+?|Total \(end-to-end\))\s+([\d.]+)\s+[\d.]+\s*$", stdout, re.M):
        run[name] = float(avg)
    return run

# ---- Ground truth: A, B là hai vùng cắt (phủ hết chiều cao) của cùng một panorama -> H_gt là phép tịnh tiến ----
def load_gray(path, height=None):
    img = PILImage.open(path).convert("L")
    if height:
        img = img.resize((round(img.width * height / img.height), height), PILImage.LANCZOS)
    return np.asarray(img, dtype=np.float64)

def locate(pano, tmpl):
    """Góc trái-trên (x, y) của tmpl trong pano: vét cạn từng pixel, chọn vị trí có SSD nhỏ nhất."""
    (H, W), (h, w) = pano.shape, tmpl.shape
    cost = {(x, y): np.mean((pano[y:y + h, x:x + w] - tmpl) ** 2)
            for y in range(H - h + 1) for x in range(W - w + 1)}
    return np.array(min(cost, key=cost.get), dtype=float)

def bilinear(img, x, y):
    x0, y0 = np.floor(x).astype(int), np.floor(y).astype(int)
    ax, ay = x - x0, y - y0
    return ((1 - ay) * ((1 - ax) * img[y0, x0] + ax * img[y0, x0 + 1])
            + ay * ((1 - ax) * img[y0 + 1, x0] + ax * img[y0 + 1, x0 + 1]))

def refine_translation(A, B, t, radius):
    """Tịnh tiến t sao cho B(p) ≈ A(p + t) trên toàn vùng chồng lấp: chọn pixel nguyên có SSD nhỏ nhất
    trong ±radius quanh t, rồi lặp Lucas–Kanade để đạt độ chính xác dưới pixel."""
    ys, xs = np.mgrid[0:B.shape[0], 0:B.shape[1]].astype(float)
    def overlap(t, margin=2):  # pixel của B rơi vào trong A (chừa biên cho nội suy và gradient)
        return ((xs + t[0] >= margin) & (xs + t[0] <= A.shape[1] - 1 - margin)
                & (ys + t[1] >= margin) & (ys + t[1] <= A.shape[0] - 1 - margin))
    sub = (xs % 3 == 0) & (ys % 3 == 0)  # lấy thưa cho bước tìm pixel nguyên
    def ssd(t):
        m = overlap(t) & sub
        return np.mean((A[(ys[m] + t[1]).astype(int), (xs[m] + t[0]).astype(int)] - B[m]) ** 2)
    t = np.round(t)
    t = min((t + (dx, dy) for dy in range(-radius, radius + 1) for dx in range(-radius, radius + 1)), key=ssd)
    for _ in range(50):
        m = overlap(t)
        xa, ya = xs[m] + t[0], ys[m] + t[1]
        r = bilinear(A, xa, ya) - B[m]
        gx = (bilinear(A, xa + 1, ya) - bilinear(A, xa - 1, ya)) / 2
        gy = (bilinear(A, xa, ya + 1) - bilinear(A, xa, ya - 1)) / 2
        step = np.linalg.solve([[gx @ gx, gx @ gy], [gx @ gy, gy @ gy]], [-(gx @ r), -(gy @ r)])
        t = t + step
        if np.abs(step).max() < 1e-4:
            break
    return t

def ground_truth_homography(pano_path, a_path, b_path):
    """H_gt (B -> A): dò thô vị trí A, B trong panorama, rồi tinh chỉnh dưới pixel trên chính A, B."""
    # Dò thô không cần ảnh lớn: thu nhỏ panorama (nếu cao hơn 360 px) để bước vét cạn chạy nhanh
    pano = load_gray(pano_path, min(PILImage.open(pano_path).height, 360))
    A, B = load_gray(a_path), load_gray(b_path)
    k = pano.shape[0] / A.shape[0]  # tỉ lệ panorama / ảnh (A, B phủ hết chiều cao panorama)
    t0 = (locate(pano, load_gray(b_path, pano.shape[0])) - locate(pano, load_gray(a_path, pano.shape[0]))) / k
    t = refine_translation(A, B, t0, radius=int(np.ceil(1 / k)) + 2)
    return np.array([[1, 0, t[0]], [0, 1, t[1]], [0, 0, 1]])

# ---- Độ đo sai số của H_V0 so với H_gt ----
def project(H, pts):
    q = np.c_[pts, np.ones(len(pts))] @ H.T
    return q[:, :2] / q[:, 2:]

def homography_errors(H_est, H_gt, B_size, A_size):
    """Sai số (pixel ảnh A) khi chiếu các điểm của ảnh B bằng H_est thay cho H_gt."""
    (w, h), (wA, hA) = B_size, A_size
    corners = np.array([[0, 0], [w - 1, 0], [0, h - 1], [w - 1, h - 1]], dtype=float)
    d = np.linalg.norm(project(H_est, corners) - project(H_gt, corners), axis=1)
    grid = np.stack(np.meshgrid(np.linspace(0, w - 1, 160), np.linspace(0, h - 1, 90)), -1).reshape(-1, 2)
    gt = project(H_gt, grid)
    inside = (gt[:, 0] >= 0) & (gt[:, 0] <= wA - 1) & (gt[:, 1] >= 0) & (gt[:, 1] <= hA - 1)
    e = np.linalg.norm(project(H_est, grid[inside]) - gt[inside], axis=1)
    return {"ACE (px)": d.mean(), "Max corner (px)": d.max(),
            "RMSE overlap (px)": np.sqrt(np.mean(e ** 2))}

Cell dưới chạy V0 trên bộ ground truth `GT_NAME` ở cả ba độ phân giải. Muốn đánh giá bộ khác chỉ cần đổi `GT_NAME`: cell tự tìm panorama `synthetic/<GT_NAME>.png` (hoặc `.jpg`) và cặp ảnh `synthetic/<độ phân giải>/<GT_NAME>/`. Với cặp `SHOW_RES`, cell hiển thị ảnh A, ảnh B, panorama ground truth, panorama V0 **không crop** và **có crop**. Sau đó cell in bảng độ lỗi của cả ba độ phân giải.

In [ ]:
GT_NAME = "GT1"                                      # tên bộ ground truth cần đánh giá
SHOW_RES = "medium"                                  # độ phân giải dùng để hiển thị ảnh

SYNTHETIC_DIR = "/content/dataset/synthetic"
GT_PAIR = os.path.join(SYNTHETIC_DIR, "{}", GT_NAME)  # A, B của từng độ phân giải: synthetic/<res>/<GT_NAME>/
found = [p for p in glob.glob(os.path.join(SYNTHETIC_DIR, GT_NAME + ".*"))
         if p.lower().endswith((".png", ".jpg", ".jpeg"))]
assert len(found) == 1, f"Cần đúng 1 ảnh {GT_NAME}.png/.jpg trong {SYNTHETIC_DIR} (đang có {len(found)})"
GT_PANORAMA = found[0]                               # panorama gốc: A và B là hai vùng cắt của ảnh này

SYN_ERR = []
for res in RESOLUTIONS:
    pair = GT_PAIR.format(res)
    a_path, b_path = pair_images(pair)
    p = stitch(pair, f"/content/{GT_NAME}_{res}.jpg", runs=1)
    assert p.returncode == 0, p.stdout + p.stderr
    run = parse_run(p.stdout)
    H_gt = ground_truth_homography(GT_PANORAMA, a_path, b_path)
    (wA, hA), (wB, hB) = PILImage.open(a_path).size, PILImage.open(b_path).size
    SYN_ERR.append({"Độ phân giải": f"{res} ({wA}×{hA})", "Inlier / Match": f'{run["inliers"]} / {run["matches"]}',
                    **homography_errors(run["H"], H_gt, (wB, hB), (wA, hA))})

q = stitch(GT_PAIR.format(SHOW_RES), f"/content/{GT_NAME}_nocrop.jpg", runs=1, crop=False)
assert q.returncode == 0, q.stdout + q.stderr

def show_synthetic():
    a_path, b_path = pair_images(GT_PAIR.format(SHOW_RES))
    print(f"Ảnh A và ảnh B ({GT_NAME}, {SHOW_RES}):")
    display(HTML(f'<div style="display:flex; gap:32px; max-width:1400px">{figure(a_path)}{figure(b_path)}</div>'))
    print("\nPanorama ground truth:")
    display(HTML(f'<div style="max-width:1400px">{figure(GT_PANORAMA, 1600, f"Ground truth {os.path.basename(GT_PANORAMA)}")}</div>'))
    print("\nPanorama V0 (không crop, giữ vùng đen):")
    display(HTML(f'<div style="max-width:1400px">{figure(f"/content/{GT_NAME}_nocrop.jpg", 1600, "V0 – không crop")}</div>'))
    print("\nPanorama V0 (crop về hình chữ nhật lớn nhất không có pixel đen):")
    display(HTML(f'<div style="max-width:1400px">{figure(f"/content/{GT_NAME}_{SHOW_RES}.jpg", 1600, "V0 – có crop")}</div>'))

    print("\nĐộ lỗi của H_V0 so với H_gt (pixel):")
    display(pd.DataFrame(SYN_ERR).style.hide(axis="index").format(precision=3)
            .set_table_styles([{"selector": "th, td", "props": "padding: 6px 24px; text-align: center"}]))

if not EMBEDDED:
    show_synthetic()

## 6. Chạy toàn bộ dataset

Chạy V0 trên mọi cặp ảnh ở các độ phân giải trong `SIZES`:
- **Cặp chuẩn:** 1 lần warm-up + `RUNS` lần đo, dùng cho bảng thời gian.
- **Cặp thử thách:** chạy 1 lần, chỉ dùng để phân tích thất bại, không đưa vào bảng thời gian.

Kết quả lưu trong `ROWS` và được các mục 7–10 dùng lại, nên chỉ cần chạy một lần. Panorama được lưu vào `/content/out/<độ phân giải>/`. Ảnh large chạy khá lâu trên CPU của Colab; mỗi dòng in ra kèm thời gian thực của cặp đó để tiện theo dõi.

Cell đầu tiên chỉ khai báo hằng số và hàm dùng chung, không chạy V0.

In [7]:
import numpy as np
import pandas as pd

SIZES = ["small", "medium", "large"]                      # các độ phân giải cần chạy
DIMS = {"small": (1280, 720), "medium": (1920, 1080), "large": (3840, 2160)}
RES_LABEL = {s: f"{s} ({w}×{h})" for s, (w, h) in DIMS.items()}

STAGE_NAMES = ["1. Preprocessing", "2. Keypoint detection", "3. Description",
               "4. Matching", "5. Homography (RANSAC)", "6. Warping & blending"]   # tên giai đoạn trong output của ./stitch
STAGE_COLS = ["S1 Tiền xử lý", "S2 Keypoint", "S3 Mô tả", "S4 Ghép cặp", "S5 RANSAC", "S6 Warp + blend"]
TOTAL = "Total (end-to-end)"
STITCH_CMD = ["./stitch"]
TABLE_STYLE = [{"selector": "th, td", "props": "padding: 6px 16px; text-align: center; white-space: nowrap"}]

def show_table(df, fmt=None):
    """Hiển thị bảng: ẩn cột chỉ mục (mọi tiêu đề nằm cùng một hàng), căn giữa và giãn cột."""
    sty = df.style.hide(axis="index").format(precision=2, na_rep="–")
    if fmt:   # định dạng riêng chỉ áp lên các cột được chỉ định, các cột khác giữ 2 chữ số thập phân
        sty = sty.format(fmt, subset=list(fmt), na_rep="–")
    display(sty.set_table_styles(TABLE_STYLE))

def parse_output(text):
    """Đọc thống kê lần warm-up, ma trận H, kích thước panorama và thời gian (avg, min) từ output của ./stitch."""
    r = {}
    if m := re.search(r"(\d+) keypoints in A, (\d+) in B", text):
        r["kp_a"], r["kp_b"] = map(int, m.groups())
    if m := re.search(r"Stage 4 \(matching\)\s*:\s*(\d+) matches", text):
        r["matches"] = int(m[1])
    if m := re.search(r"Stage 5 \(RANSAC\)\s*:\s*(\d+) inliers", text):
        r["inliers"] = int(m[1])
    if m := re.search(r"H \(B -> A\)\s*:\s*\[([^\]]*)\]", text):
        r["H"] = np.array(m[1].replace(";", " ").split(), dtype=float).reshape(3, 3)
    if m := re.search(r"Panorama (\d+)x(\d+) saved", text):
        r["pano"] = f"{m[1]}×{m[2]}"
    for name in STAGE_NAMES + [TOTAL]:
        if m := re.search(rf"^{re.escape(name)}\s+([\d.]+)\s+([\d.]+)\s*$", text, re.M):
            r[name], r[name + " min"] = float(m[1]), float(m[2])
    return r

def run_v0(left, right, out, runs=RUNS, crop=True):
    args = [left, right, out, "--version", "0", "--runs", str(runs)] + ([] if crop else ["--no-crop"])
    p = subprocess.run(STITCH_CMD + args, capture_output=True, text=True)
    r = parse_output(p.stdout)
    r["ok"] = p.returncode == 0 and TOTAL in r
    r["log"] = p.stdout + p.stderr
    return r

def error_reason(r):
    """Dòng lỗi cụ thể của V0 (ví dụ RANSAC thất bại), nếu không có thì dòng cuối của log."""
    lines = r["log"].strip().splitlines()
    return next((l.strip() for l in lines if "[V0 Error]" in l), lines[-1] if lines else "")

In [ ]:
import time

OUT_DIR = "/content/out"

def pair_info(d):
    """(độ phân giải, nhóm, tên cặp) từ đường dẫn, ví dụ ('medium', 'standard', 'Indoor / Pair 1')."""
    parts = os.path.relpath(d, "/content/dataset").split(os.sep)
    if len(parts) < 3:
        return parts[0], "", ""
    group = "challenge" if "challenge" in parts[1].lower() else "standard"
    return parts[0], group, " / ".join((["Challenge"] if group == "challenge" else []) + parts[2:])

ROWS = []
jobs = [(pair_info(d), d) for d in PAIR_DIRS if pair_info(d)[0] in SIZES]
jobs.sort(key=lambda j: (SIZES.index(j[0][0]), j[0][1] != "standard", j[0][2]))
for (size, group, label), d in jobs:
    out = os.path.join(OUT_DIR, size, re.sub(r"\W+", "_", label) + ".jpg")
    os.makedirs(os.path.dirname(out), exist_ok=True)
    t0 = time.time()
    r = run_v0(*pair_images(d), out, runs=RUNS if group == "standard" else 1)   # cặp thử thách không vào bảng thời gian
    r.update(size=size, group=group, pair=label, dir=d, out=out)
    ROWS.append(r)
    status = f"{r[TOTAL]:9.1f} ms" if r["ok"] else "   THẤT BẠI"
    print(f"{RES_LABEL[size]:19s} {label:20s} {status} | keypoint {r.get('kp_a', '–')}/{r.get('kp_b', '–')}, "
          f"cặp ghép {r.get('matches', '–')}, inlier {r.get('inliers', '–')}  ({time.time() - t0:.0f}s)")

failed = [r for r in ROWS if not r["ok"]]
print(f"\n{len(ROWS) - len(failed)}/{len(ROWS)} lần chạy thành công")
for r in failed:
    print(f"  THẤT BẠI {RES_LABEL[r['size']]} – {r['pair']}: {error_reason(r)}")

## 7. Panorama đầu ra (Mục 5.2)

Panorama của mọi cặp ở độ phân giải `SHOW_SIZE`, kèm số keypoint, số cặp ghép và số inlier. Với cặp thất bại, cell in thông báo lỗi và hiển thị hai ảnh đầu vào để phục vụ phần phân tích thất bại.

In [ ]:
SHOW_SIZE = "medium"   # độ phân giải dùng để hiển thị panorama

def show_panoramas(size=SHOW_SIZE):
    print(f"Panorama V0 ở độ phân giải {RES_LABEL[size]}:")
    for r in (x for x in ROWS if x["size"] == size):
        info = (f"{r['pair']} – keypoint {r.get('kp_a', '–')}/{r.get('kp_b', '–')}, "
                f"cặp ghép {r.get('matches', '–')}, inlier {r.get('inliers', '–')}")
        if r["ok"]:
            display(HTML(f'<div style="max-width:1200px; margin-bottom:24px">{figure(r["out"], 1400, info)}</div>'))
        else:
            left, right = pair_images(r["dir"])
            print(f"\nTHẤT BẠI – {info}\n  {error_reason(r)}")
            display(HTML(f'<div style="display:flex; gap:32px; max-width:1200px">{figure(left)}{figure(right)}</div>'))

if not EMBEDDED:
    show_panoramas()

## 8. Bảng thời gian từng giai đoạn (Mục 5.3)

Thời gian trung bình của `RUNS` lần đo (ms), không gồm đọc/ghi ảnh.
- **Bảng 1:** trung bình trên các **cặp chuẩn** chạy thành công, mỗi cột là một độ phân giải.
- **Bảng 2:** chi tiết từng cặp. Cột *Keypoint* là trung bình số keypoint của A và B, dùng để giải thích thời gian của Giai đoạn 3–4.

Toàn bộ số liệu được lưu vào `/content/v0_timing.csv`.

In [ ]:
import platform

try:
    cpu = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
except (OSError, StopIteration):
    cpu = platform.processor() or "không rõ"

df = pd.DataFrame([{"size": r["size"], "Độ phân giải": RES_LABEL[r["size"]], "Cặp": r["pair"],
                    "Keypoint": (r["kp_a"] + r["kp_b"]) / 2, "Cặp ghép": r["matches"], "Inlier": r["inliers"],
                    **{c: r[s] for c, s in zip(STAGE_COLS, STAGE_NAMES)}, "Tổng": r[TOTAL], "Tổng (min)": r[TOTAL + " min"]}
                   for r in ROWS if r["ok"] and r["group"] == "standard"])
df.drop(columns="size").to_csv("/content/v0_timing.csv", index=False)

# SUMMARY: trung bình trên các cặp chuẩn, chỉ mục là small / medium / large (dùng cho các phần sau)
SUMMARY = df.groupby("size")[["Keypoint", "Cặp ghép"] + STAGE_COLS + ["Tổng"]].mean()
SUMMARY.insert(0, "Số cặp", df.groupby("size").size())
SUMMARY = SUMMARY.reindex([s for s in SIZES if s in SUMMARY.index])

def show_timing():
    print(f"CPU: {cpu} | V0 đơn luồng | {RUNS} lần đo + 1 lần warm-up | không tính đọc/ghi ảnh\n")
    print("Thời gian trung bình mỗi giai đoạn trên các cặp chuẩn (ms):")
    by_stage = SUMMARY[STAGE_COLS + ["Tổng"]].T.rename(columns=RES_LABEL)
    by_stage.insert(0, "Giai đoạn", by_stage.index)
    show_table(by_stage)
    print("Số cặp: " + ", ".join(f"{RES_LABEL[s]}: {n}" for s, n in SUMMARY["Số cặp"].items()))

    print("\nChi tiết từng cặp chuẩn (ms):")
    show_table(df.drop(columns="size"), {"Keypoint": "{:.0f}"})
    print("Keypoint là trung bình số keypoint của A và B; toàn bộ số liệu được lưu tại /content/v0_timing.csv")

if not EMBEDDED:
    show_timing()

## 9. Giai đoạn chiếm nhiều thời gian nhất (Mục 5.4)

Tỉ lệ (%) thời gian của từng giai đoạn ở mỗi độ phân giải, và mức tăng thời gian so với small. Đối chiếu mức tăng với số pixel (×2.25 và ×9) và với bình phương số keypoint (khối lượng của ghép cặp vét cạn). Từ đó thấy giai đoạn nào tăng theo kích thước ảnh, giai đoạn nào tăng theo số keypoint.

In [ ]:
import matplotlib.pyplot as plt

share = SUMMARY[STAGE_COLS].div(SUMMARY[STAGE_COLS].sum(axis=1), axis=0) * 100
growth = None
if "small" in SUMMARY.index:
    growth = SUMMARY[STAGE_COLS + ["Tổng"]].div(SUMMARY.loc["small", STAGE_COLS + ["Tổng"]])
    growth.insert(0, "Pixel", [DIMS[s][0] * DIMS[s][1] / (DIMS["small"][0] * DIMS["small"][1]) for s in growth.index])
    growth.insert(1, "Keypoint²", (SUMMARY["Keypoint"] / SUMMARY.loc["small", "Keypoint"]) ** 2)

def show_hot_stages():
    print("Tỉ lệ thời gian của từng giai đoạn (%):")
    show_table(share.rename(index=RES_LABEL).rename_axis("Độ phân giải").reset_index(),
               {c: "{:.1f}%" for c in STAGE_COLS})
    for size, row in share.iterrows():
        top = row.sort_values(ascending=False)
        print(f"  {RES_LABEL[size]}: {top.index[0]} chiếm {top.iloc[0]:.1f}%, {top.index[1]} chiếm {top.iloc[1]:.1f}%")
    if growth is not None:
        print("\nMức tăng so với small (lần):")
        show_table(growth.rename(index=RES_LABEL).rename_axis("Độ phân giải").reset_index(),
                   {c: "×{:.2f}" for c in growth.columns})

    fig, ax = plt.subplots(figsize=(11, 0.9 * len(share) + 1.6))
    left = np.zeros(len(share))
    labels = [RES_LABEL[s] for s in share.index]
    for i, col in enumerate(STAGE_COLS):
        bars = ax.barh(labels, share[col], left=left, height=0.6, color=plt.cm.tab10(i), label=col)
        for bar, v in zip(bars, share[col]):
            if v >= 4:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2, f"{v:.0f}%",
                        ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        left += share[col].to_numpy()
    ax.set_xlim(0, 100)
    ax.set_xlabel("% thời gian V0")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(ncol=6, bbox_to_anchor=(0.5, -0.25), loc="upper center", frameon=False, fontsize=9)
    ax.set_title("Phân rã thời gian V0 theo giai đoạn")
    plt.tight_layout()
    plt.show()

if not EMBEDDED:
    show_hot_stages()

## 10. Ước lượng theo định luật Amdahl (Mục 5.5)

Nếu các giai đoạn được tăng tốc chiếm tỉ lệ $p$ thời gian của V0 và chạy nhanh hơn $s$ lần, thì speedup end-to-end là

$$S(s) = \frac{1}{(1 - p) + p / s} \;\le\; \frac{1}{1 - p}.$$

Bảng tính $p$, speedup khi GPU nhanh ×10 và ×50, và giới hạn $1/(1-p)$ cho bốn kịch bản: chỉ tăng tốc giai đoạn nặng nhất, hai giai đoạn nặng nhất, mọi giai đoạn trừ RANSAC, và cả 6 giai đoạn. Ở kịch bản cuối $p = 1$ trong phạm vi V0 đo được. Khi đó giới hạn thực tế đến từ thời gian truyền dữ liệu host↔device và phần việc còn lại trên CPU, sẽ đo ở V1.

Biểu đồ vẽ $S(s)$ ở độ phân giải benchmark chính (medium), trục log, nét đứt là giới hạn của từng kịch bản.

In [ ]:
def amdahl(p, s):
    """Speedup end-to-end khi phần chiếm tỉ lệ p được tăng tốc s lần (s = ∞: giới hạn 1 / (1 − p))."""
    return 1 / (1 - p) if np.isinf(s) else 1 / ((1 - p) + p / s)

rows = []
for size, t in SUMMARY[STAGE_COLS].iterrows():
    ranked = list(t.sort_values(ascending=False).index)
    scenarios = {"Giai đoạn nặng nhất": ranked[:1], "2 giai đoạn nặng nhất": ranked[:2],
                 "Trừ RANSAC (S5)": [c for c in STAGE_COLS if c != "S5 RANSAC"], "Cả 6 giai đoạn": STAGE_COLS}
    for name, cols in scenarios.items():
        p = 1.0 if len(cols) == len(STAGE_COLS) else t[cols].sum() / t.sum()
        rows.append({"Độ phân giải": RES_LABEL[size], "Kịch bản": name,
                     "Giai đoạn tăng tốc": ", ".join(c.split()[0] for c in cols), "p": p,
                     "GPU ×10": amdahl(p, 10), "GPU ×50": amdahl(p, 50),
                     "Giới hạn 1/(1−p)": np.inf if p >= 1 else amdahl(p, np.inf)})
AMDAHL = pd.DataFrame(rows)
BENCH = "medium" if "medium" in SUMMARY.index else SUMMARY.index[0]

def show_amdahl():
    speed = lambda v: "∞" if np.isinf(v) else f"{v:.2f}×"
    print("Speedup end-to-end theo định luật Amdahl (GPU ×s: các giai đoạn được chọn nhanh hơn s lần):")
    show_table(AMDAHL, {"p": "{:.1%}", "GPU ×10": speed, "GPU ×50": speed, "Giới hạn 1/(1−p)": speed})

    s = np.logspace(0, 3, 200)
    fig, ax = plt.subplots(figsize=(9, 5))
    for i, r in enumerate(AMDAHL[AMDAHL["Độ phân giải"] == RES_LABEL[BENCH]].itertuples()):
        if r.p >= 1:
            continue   # p = 1: speedup tăng tuyến tính theo s, không có giới hạn
        limit = 1 / (1 - r.p)
        ax.plot(s, [amdahl(r.p, x) for x in s], color=plt.cm.tab10(i), linewidth=2,
                label=f"{r._2}: p = {r.p:.1%}, tối đa {limit:.1f}×")
        ax.axhline(limit, color=plt.cm.tab10(i), linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yscale("log")   # thang log để thấy rõ cả giới hạn nhỏ (vài lần) lẫn lớn (hàng trăm lần)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×"))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}×"))
    ax.set_xlabel("Mức tăng tốc s của các giai đoạn được chọn")
    ax.set_ylabel("Speedup end-to-end")
    ax.set_title(f"Định luật Amdahl – {RES_LABEL[BENCH]} (nét đứt: giới hạn 1/(1−p))")
    ax.spines[["top", "right"]].set_visible(False)
    # Chú thích đặt dưới trục hoành, ngoài vùng vẽ, để không đè lên đường cong và đường giới hạn
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=1)
    plt.tight_layout()
    plt.show()

if not EMBEDDED:
    show_amdahl()